# Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
 
import os

from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.vector_db import VectorStore
from dotenv import load_dotenv
import chromadb

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
BASE_URL = os.getenv("BASE_URL")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [2]:
from chromadb.utils import embedding_functions

chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
  api_base=os.getenv("BASE_URL"),
  api_key=os.getenv("CHROMA_OPENAI_API_KEY")
)
collection = chroma_client.get_or_create_collection(
   name="udaplay",
   embedding_function=embedding_fn
)

vector_store = VectorStore(collection)

@tool
def retrieve_game(query: str):
  """
  Tool Docstring:
    Performs a semantic search against the internal game knowledge base (Vector DB)
    to find the most relevant game records for the given query.
    Use this tool first before searching the web.
    args:
    - query: a natural language question or topic about the video game industry.

    Returns up to 5 matching results. Each result contains:
    - documents: the raw text content of the matching game record
    - metadatas: structured metadata for each match, including:
        - Name: Name of the game
        - Platform: Platform (e.g. PlayStation, Xbox 360, Game Boy)
        - YearOfRelease: Year the game was released on that platform
    - distances: similarity scores (lower = more similar)
  """
  results = vector_store.query(query, n_results=5)
  return results


#### Evaluate Retrieval Tool

In [3]:
from lib.evaluation import AgentEvaluator, EvaluationResult
from lib.parsers import PydanticOutputParser
from lib.llm import LLM

evaluator = AgentEvaluator()
# Override the judge to use the proxy base_url and api_key
evaluator.llm_judge = LLM(model="gpt-4o-mini", api_key=OPENAI_API_KEY, base_url=BASE_URL)

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> EvaluationResult:
  """
  Tool Docstring:
    Uses an LLM as judge to evaluate whether the retrieved documents are sufficient
    to answer the user's question.
    args:
    - question: original question from the user
    - retrieved_docs: list of documents retrieved from the Vector Database
    The result is an EvaluationResult containing:
    - task_completion: whether the documents allow the query to be fully answered
    - quality_control: whether the documents are well-structured and relevant
    - tool_interaction: assessment of the retrieval tool's output usefulness
    - system_metrics: token and performance metrics
    - overall_score: a score between 0 and 1 reflecting document usefulness
    - feedback: detailed explanation to decide whether to accept or search the web
  """
  judge_prompt = f"""Your task is to evaluate if the documents are enough to respond the query. 
Give a detailed explanation, so it's possible to take an action to accept it or not.

Query: {question}

Retrieved Documents:
{retrieved_docs}

Evaluate whether the above documents contain sufficient information to answer the query.
Fill in the evaluation result fields based on your analysis:
- task_completion: whether the documents allow the query to be answered, steps_taken=1
- quality_control: format_correct=true if documents are well-structured, instructions_followed based on relevance
- tool_interaction: correct_tool_selected=true, valid_arguments=true, tool_result_useful based on relevance
- system_metrics: total_tokens=0, execution_time=0.0, tool_call_latency=0.0
- overall_score: a score between 0 and 1 reflecting how useful the documents are
- feedback: detailed explanation of the evaluation
"""
  response = evaluator.llm_judge.invoke(
      input=judge_prompt,
      response_format=EvaluationResult
  )
  parser = PydanticOutputParser(model_class=EvaluationResult)
  return parser.parse(response)


#### Game Web Search Tool

In [4]:
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(query: str):
  """
  Tool Docstring:
    Performs a web search using the Tavily API to find recent information about the video game industry.
    Use this tool if the retrieved documents from the Vector DB are not sufficient to answer the user's query.
    args:
    - query: a natural language question or topic about the video game industry.

    Returns up to 5 relevant search results, each containing:
    - url: the URL of the web page
    - content: the page content
  """
  results = tavily_client.get_search_context(query, search_depth="advanced", max_results=5)
  return results

### Agent

In [5]:
import json
from pydantic import BaseModel
from typing import List, Optional, TypedDict, Union

from lib.llm import LLM
from lib.state_machine import StateMachine, Step, EntryPoint, Termination  
from lib.tooling import ToolCall

tools = [retrieve_game, evaluate_retrieval, game_web_search]

# define agent state schema
class AgentState(TypedDict):
    user_query: str  # The current user query being processed
    instructions: str  # System instructions for the agent
    messages: List[dict]  # List of conversation messages
    current_tool_calls: Optional[List[ToolCall]]  # Current pending tool calls

# define our steps
def prepare_messages_step(state: AgentState) -> AgentState:
    """Step logic: Prepare messages for LLM consumption"""

    messages = [
        SystemMessage(content=state["instructions"]),
        UserMessage(content=state["user_query"])
    ]
    
    return {
        "messages": messages
    }

def llm_step(state: AgentState) -> AgentState:
    """Step logic: Process the current state through the LLM"""

    llm = LLM(
        model="gpt-4o-mini",
        temperature=0.3,
        tools=tools,
        api_key=OPENAI_API_KEY,
        base_url=BASE_URL,
    )

    response = llm.invoke(state["messages"])
    tool_calls = response.tool_calls if response.tool_calls else None

    ai_message = AIMessage(content=response.content, tool_calls=tool_calls)
    
    return {
        "messages": state["messages"] + [ai_message],
        "current_tool_calls": tool_calls
    }

def serialize_result(result) -> str:
    """Serialize tool result to JSON string, handling Pydantic models."""
    if isinstance(result, BaseModel):
        return result.model_dump_json()
    return json.dumps(result)

def tool_step(state: AgentState) -> AgentState:
    """Step logic: Execute any pending tool calls"""
    tool_calls = state["current_tool_calls"] or []
    tool_messages = []
    
    for call in tool_calls:
        function_name = call.function.name
        function_args = json.loads(call.function.arguments)
        tool_call_id = call.id
        tool = next((t for t in tools if t.name == function_name), None)
        if tool:
            result = tool(**function_args)
            tool_messages.append(
                ToolMessage(
                    content=serialize_result(result), 
                    tool_call_id=tool_call_id, 
                    name=function_name, 
                )
            )
    
    return {
        "messages": state["messages"] + tool_messages,
        "current_tool_calls": None
    }

# now we can define our state machine
agent_state_machine = StateMachine[AgentState](AgentState)

entry = EntryPoint[AgentState]()
message_prep = Step[AgentState]("message_prep", prepare_messages_step)
llm_processor = Step[AgentState]("llm_processor", llm_step)
tool_executor = Step[AgentState]("tool_executor", tool_step)
termination = Termination[AgentState]()

agent_state_machine.add_steps(
    [entry, message_prep, llm_processor, tool_executor, termination]
)

agent_state_machine.connect(entry, message_prep)
agent_state_machine.connect(message_prep, llm_processor)

def check_tool_calls(state: AgentState) -> Union[Step[AgentState], str]:
    if state.get("current_tool_calls"):
        return tool_executor
    return termination

agent_state_machine.connect(
    source=llm_processor, 
    targets=[tool_executor, termination], 
    condition=check_tool_calls
)

agent_state_machine.connect(source=tool_executor, targets=llm_processor)

agent_instructions = """You are UdaPlay, an AI Research Agent for the video game industry.
Follow this workflow for every query:
1. Use retrieve_game to search the internal knowledge base first.
2. Use evaluate_retrieval to assess whether the retrieved documents are sufficient.
3. If the evaluation score is low or feedback indicates insufficient data, use game_web_search to find the answer online.
4. Provide a clear, concise answer to the user.
5. You MUST cite your sources at the end of every answer (e.g., [Source: Internal Database] or provide the web URL)."""

def invoke_agent(query: str, messages: List[dict] = []) -> AgentState:
    """Function to run the agent state machine"""
        
    initial_state = AgentState(
        user_query=query,
        instructions=agent_instructions,
        messages=messages,
        current_tool_calls=None
    )
    return agent_state_machine.run(initial_state)


In [6]:
def print_agent_trace(agent_response):
    """Print a readable trace of the agent's reasoning and tool usage."""
    state = agent_response.get_final_state()
    for msg in state["messages"]:
        if getattr(msg, 'tool_calls', None):
            print(f"🛠️  Agent called tool(s): {[call.function.name for call in msg.tool_calls]}")
        elif msg.role == "tool":
            snippet = msg.content[:200] + "..." if len(msg.content) > 200 else msg.content
            print(f"📥 Tool returned data: {snippet}")
        elif msg.role == "assistant" and msg.content:
            print(f"🤖 Final Answer: {msg.content}")
    print("-" * 60)
    return state["messages"]

# - When Pokémon Gold and Silver was released?
print("Query 1: When Pokémon Gold and Silver was released?")
first_query = "When Pokémon Gold and Silver was released?"
agent_response = invoke_agent(first_query)
messages = print_agent_trace(agent_response)

# - Which one was the first 3D platformer Mario game?
print("Query 2: Which one was the first 3D platformer Mario game?")
second_query = "Which one was the first 3D platformer Mario game?"
agent_response = invoke_agent(second_query, messages=messages)
messages = print_agent_trace(agent_response)

# - Was Mortal Kombat X released for Playstation 5?
print("Query 3: Was Mortal Kombat X released for Playstation 5?")
third_query = "Was Mortal Kombat X released for Playstation 5?"
agent_response = invoke_agent(third_query, messages=messages)
messages = print_agent_trace(agent_response)


Query 1: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
🛠️  Agent called tool(s): ['retrieve_game']
📥 Tool returned data: {"ids": [["006", "007", "012", "009", "008"]], "embeddings": null, "documents": [["[Game Boy Color] Pok\u00e9mon Gold and Silver (1999) - Second-generation Pok\u00e9mon games introducing new regions, ...
🛠️  Agent called tool(s): ['evaluate_retrieval']
📥 Tool returned data: {"task_completion":{"task_completed":true,"steps_taken":1,"expected_steps":1},"quality_control":{"format_correct":true,"instructions_followed":true},"tool_interaction":{"correct_tool_selected":true,"v...
🤖 Final Answer: Pokémon Gold and Silver were released i

/tmp/ipykernel_51032/1114793863.py:17: DeprecationWarning: get_search_context is deprecated and will be removed in future versions.
  results = tavily_client.get_search_context(query, search_depth="advanced", max_results=5)


[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
🛠️  Agent called tool(s): ['retrieve_game']
📥 Tool returned data: {"ids": [["005", "004", "015", "003", "012"]], "embeddings": null, "documents": [["[PlayStation 5] Marvel's Spider-Man 2 (2023) - The sequel to the acclaimed Spider-Man game, featuring both Peter Park...
🛠️  Agent called tool(s): ['evaluate_retrieval']
📥 Tool returned data: {"task_completion":{"task_completed":false,"steps_taken":1,"expected_steps":1},"quality_control":{"format_correct":true,"instructions_followed":false},"tool_interaction":{"correct_tool_selected":true,...
🛠️  Agent called tool(s): ['game_web_search']
📥 Tool returned data: "[{\"url\": \"https://store.playstation.com/en-us/product/UP1018-CUSA00967_00-MORTALKOMBATX000\", \"content\": \"To play this game on PS5, your system may need to be updated to the latest system softw...
🤖 Final Answer: Mortal Kombat X was not released 